In [1]:
# Standard libraries
import requests

# Classification
from agent1.classification.ensemble import IntentClassifierEnsemble

# Routing
from agent1.routing.intent_mapping import INTENT_TO_ACTION
from agent1.routing.intent_router import IntentRouter

# Conversation
from agent1.conversation import (
    ConversationState,
    ContextBuilder,
)



In [2]:
# Analysis
from agent1.analysis import (
    RequestAnalyzer,
)

In [3]:
ensemble_model = IntentClassifierEnsemble.load(
    "../models/intent_classifier/ensemble.joblib"
)

In [ ]:
# if from project root,use
# ensemble_model = IntentClassifierEnsemble.load(
#     "models/intent_classifier/ensemble.joblib"
# )

In [4]:
# initializing the router
router = IntentRouter(
    intent_to_action=INTENT_TO_ACTION,
    min_action_score=0.40,
    min_action_margin=0.10,
    top_k=5,
)

In [5]:
def ollama_llm(
    system_prompt: str,
    user_prompt: str,
) -> str:

    response = requests.post(
        "http://localhost:11434/api/chat",
        json={
            # "model": "qwen3:4b",
            "model": "llama3.2",
            "messages": [
                {
                    "role": "system",
                    "content": system_prompt,
                },
                {
                    "role": "user",
                    "content": user_prompt,
                },
            ],
            "stream": False,
            "format": "json",
        },
        timeout=120,
    )

    response.raise_for_status()

    return response.json()["message"]["content"]

In [6]:
from agent1.analysis.request_type_normalizer import (
    RequestTypeNormalizer,
)

In [7]:
# request_type_normalizer = RequestTypeNormalizer(
#     encode=semantic_model.encode,
#     threshold=0.60,
# )

# analyzer = RequestAnalyzer(
#     llm=llm,
#     request_type_normalizer=request_type_normalizer,
# )

request_type_normalizer = RequestTypeNormalizer(
    encode=ensemble_model.semantic_classifier.encode,
    threshold=0.60,
)

analyzer = RequestAnalyzer(
    llm=ollama_llm,
    request_type_normalizer=request_type_normalizer,
)

In [8]:
context_builder = ContextBuilder(
    max_messages=6
)

In [ ]:
# pull first!!!
# ollama pull qwen3:4b
# ollama pull llama3.2

In [ ]:
# actual test
import numpy as np
top_n_intents = 5

query = (
    "The item I bought isn't quite what I expected. "
    "What are my options for getting a different one?"
)

state = ConversationState()

state.add_user_message(query)

context = context_builder.build_for_analyzer(
    state
)

probabilities = ensemble_model.predict_proba(
    [context]
)[0]

top_idx = np.argsort(probabilities)[::-1][:top_n_intents]

intent_candidates_text = "\n".join(
    f"- {ensemble_model.classes_[idx]}: {probabilities[idx]:.4f}"
    for idx in top_idx
)

routing_decision = router.route(
    probabilities,
    ensemble_model.classes_,
)

action_plan = analyzer.analyze(
    conversation_context=context,
    routing_decision=routing_decision,
    intent_candidates_text=intent_candidates_text,
)

print("CONTEXT:")
print(context)

print("\nROUTING:")
print(routing_decision)

print("\nACTION PLAN:")
print(
    action_plan.model_dump_json(
        indent=2
    )
)

In [9]:
# several tests
test_queries = [
    "Where is my order?",

    (
        "Cancel my current order and tell me "
        "whether the blue version is available."
    ),

    "Something is wrong with my order.",

    (
        "The item I bought isn't quite what I expected. "
        "What are my options for getting a different one?"
    ),
]

In [9]:
# new test
from agent1.agent import Agent

agent = Agent(
    ensemble_model=ensemble_model,
    router=router,
    analyzer=analyzer,
    context_builder=context_builder,
)

In [10]:
print("Agent:", agent.handle_message("Please track my order."))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Agent: Please provide one order number, such as ORD-123.


In [11]:
print(agent.last_action_plan.model_dump_json(indent=2))

{
  "request_type": "single_action",
  "intent": "track_order",
  "action": "order_service",
  "candidate_intents": [],
  "candidate_actions": [],
  "needs_clarification": false,
  "clarification_question": null,
  "reason": "The user clearly wants to track their order, as indicated by the phrase 'track my order'. The top predicted intent score is 0.7753, which is strong evidence in favor of this intent."
}


In [12]:
print("Agent:", agent.handle_message("My order is ORD-123."))

Agent: Your order ORD-123 has status: shipped.


In [13]:
# testing a new query
agent.reset()

print(
    "Agent:",
    agent.handle_message("Please track order ORD-456."),
)



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Agent: Your order ORD-456 has status: processing.
